# SAM como referencia de comparación (baseline zero-shot)

La consigna (4.2) pide ejecutar SAM **sin entrenarlo**, con el **bounding box predicho** como prompt, sobre el conjunto de prueba, y reportar Dice e IoU frente al modelo propio. SAM no forma parte del pipeline ni genera etiquetas: es solo evidencia de cómo se comporta un modelo generalista fuera de dominio.

Este cuaderno usa `create_sam_predictor` y `run_sam_on_predicted_boxes` (secciones 23.12 y 23.13 de `proyecto_pengwin.ipynb`, ya probadas con un predictor simulado) y los aplica a todos los cortes con hueso de cada caso.

**Protocolo (para que la comparación sea justa):**
- Mismas cajas para los dos: las cajas predichas por el detector propio tras el NMS (las guarda `salida_3d.ipynb`, sección 10.3, en `evidencias/salida_3d/cache/`), con el mismo corte de score que usa el pipeline.
- Mismos cortes: los que tienen hueso en la referencia.
- Misma métrica: Dice e IoU **por región** (sacro, coxal izquierdo, coxal derecho), con intersección y áreas acumuladas sobre todos los cortes. SAM con una caja por región devuelve una región, no fragmentos, así que la comparación es a nivel de región; no se compara por fragmento.
- Imagen: la misma que ve el modelo (ventana HU + Otsu/CLAHE + letterbox 256), repetida en 3 canales.

**Antes de correrlo:**
1. `pip install segment-anything` en el entorno del proyecto.
2. Descargar un checkpoint (por ejemplo `sam_vit_b_01ec64.pth`, del repositorio oficial de SAM de Meta) y dejarlo en `modelos_sam/`. Agregar `modelos_sam/` y `evidencias/salida_3d/cache/` al `.gitignore`: no se suben a GitHub.
3. Tener las cajas predichas del modelo que se va a evaluar (ejecutar `salida_3d.ipynb` hasta 10.3).


## 1. Configuración


In [ ]:
import ast, gc, json, os, sys, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from PIL import Image
from skimage.exposure import equalize_adapthist      # los usa enhance_bone_contrast (viene del cuaderno del proyecto)
from skimage.filters import threshold_otsu

warnings.filterwarnings("ignore", message="'mode' parameter is deprecated")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 256


def find_project_root(marker="proyecto_pengwin.ipynb", max_up=3):
    start = Path(os.environ.get("PENGWIN_ROOT", Path.cwd())).resolve()
    for folder in [start, *list(start.parents)[:max_up]]:
        if (folder / marker).is_file():
            return folder
    raise FileNotFoundError(f"No encuentro {marker} desde {start}. Guarda este cuaderno en la carpeta del proyecto "
                            f"o escribe ROOT = Path(r'C:\\ruta\\del\\proyecto').")


ROOT = find_project_root()
BASE_DIR = ROOT                       # lo usa resolve_repo_path (viene de proyecto_pengwin.ipynb)
EVIDENCE_DIR = ROOT / "evidencias"
SAM_DIR = EVIDENCE_DIR / "sam"
SAM_DIR.mkdir(parents=True, exist_ok=True)


def _target_names(node):
    targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
    names = set()
    for target in targets:
        if isinstance(target, ast.Name):
            names.add(target.id)
    return names


def _load_existing_definitions(notebook_path, definition_names, value_names=()):
    notebook = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    wanted_defs, wanted_values = set(definition_names), set(value_names)
    snippets, found_defs, found_values = [], set(), set()
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue
        for node in tree.body:
            name = getattr(node, "name", None)
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and name in wanted_defs and name not in found_defs:
                snippets.append(ast.get_source_segment(source, node)); found_defs.add(name)
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                selected = _target_names(node) & wanted_values
                if selected and not selected.issubset(found_values):
                    snippets.append(ast.get_source_segment(source, node)); found_values.update(selected)
    missing = (wanted_defs - found_defs) | (wanted_values - found_values)
    if missing:
        raise RuntimeError(f"Faltan definiciones en {Path(notebook_path).name}: {sorted(missing)}")
    exec("\n\n".join(snippets), globals())
    return len(snippets)


n_defs = _load_existing_definitions(
    ROOT / "proyecto_pengwin.ipynb",
    {"resolve_repo_path", "apply_hu_window", "enhance_bone_contrast", "instance_to_semantic", "letterbox_pair",
     "create_sam_predictor", "run_sam_on_predicted_boxes"},
    {"CLASS_ID_TO_NAME", "TARGET_COLORS"})
print(f"{n_defs} definiciones cargadas de proyecto_pengwin.ipynb | dispositivo: {DEVICE}")

df = pd.read_csv(ROOT / "splits" / "dataset.csv", dtype={"case_id": str})
CASES = {s: sorted(df.loc[df["split"] == s, "case_id"]) for s in ("train", "val", "test")}
print({s: len(c) for s, c in CASES.items()})


In [ ]:
SPLIT = "val"                      # "test" SOLO cuando el modelo y los parámetros estén congelados (la consigna pide evaluar test una vez)
SAM_MODEL_TYPE = "vit_b"           # vit_b es el más liviano; vit_l y vit_h necesitan más memoria
SAM_CHECKPOINT = ROOT / "modelos_sam" / "sam_vit_b_01ec64.pth"       # local; NO subir a Git (agregar modelos_sam/ al .gitignore)

BOXES_DIR = EVIDENCE_DIR / "salida_3d" / "cache"                     # salidas de salida_3d.ipynb (sección 10.3): cajas predichas tras el NMS
CONFIG_3D = EVIDENCE_DIR / "salida_3d" / "config_3d.json"
BOX_SCORE_MIN = 0.05
if CONFIG_3D.is_file():            # misma regla de corte activo que el pipeline propio
    BOX_SCORE_MIN = json.loads(CONFIG_3D.read_text(encoding="utf-8"))["config"]["decode"]["det_gate"]

SLICE_STRIDE = 1                   # 2 = un corte de cada 2 (SAM en CPU es lento; en GPU no hace falta)
MAX_SLICES_PER_CASE = None         # p. ej. 150 para una prueba rápida
ALSO_GT_BOXES = False              # True: segunda corrida con cajas REALES (cota superior de SAM); no es el protocolo de la consigna
print(f"split={SPLIT} | casos: {CASES[SPLIT]} | score mínimo de caja: {BOX_SCORE_MIN} | checkpoint existe: {SAM_CHECKPOINT.is_file()}")


`SPLIT = "val"` sirve para probar y revisar. Para el número final, cambiar a `"test"` **una sola vez**, con el modelo y los parámetros congelados. En CPU, SAM puede tardar varios segundos por corte: `SLICE_STRIDE` y `MAX_SLICES_PER_CASE` permiten una prueba rápida o un subconjunto fijo (anotar cuál se usó para que Hoyos evalúe los mismos cortes).


## 2. Funciones


In [ ]:
REGION_NAMES = {1: "sacro", 2: "coxal_izquierdo", 3: "coxal_derecho"}


class RegionAccumulator:
    """Dice e IoU por región acumulando intersección y áreas sobre todos los cortes (misma fórmula que 23.6b)."""
    def __init__(self):
        self.c = {k: [0.0, 0.0, 0.0] for k in (1, 2, 3)}      # inter, pred, gt

    def add(self, pred, gt):
        for k in (1, 2, 3):
            p, g = pred == k, gt == k
            self.c[k][0] += float((p & g).sum()); self.c[k][1] += float(p.sum()); self.c[k][2] += float(g.sum())

    def summary(self):
        out = {}
        for k, (i, p, g) in self.c.items():
            out[f"dice_{REGION_NAMES[k]}"] = 2 * i / (p + g) if p + g else np.nan
            out[f"iou_{REGION_NAMES[k]}"] = i / (p + g - i) if p + g - i else np.nan
        out["dice_medio"] = float(np.nanmean([out[f"dice_{n}"] for n in REGION_NAMES.values()]))
        out["iou_medio"] = float(np.nanmean([out[f"iou_{n}"] for n in REGION_NAMES.values()]))
        return out


def load_case(case_id):
    row = df[df["case_id"] == case_id].iloc[0]
    image = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
    label = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(row["label_path"])))).astype(np.int16)
    return sitk.GetArrayFromImage(image).astype(np.float32), label


def load_predicted_boxes(case_id):
    """Cajas predichas (tras el NMS) y semántica del modelo propio, guardadas por salida_3d.ipynb (10.3)."""
    stem = BOXES_DIR / f"bundle_{case_id}"
    if not stem.with_suffix(".json").is_file():
        raise FileNotFoundError(f"Falta {stem.with_suffix('.json')}: ejecuta salida_3d.ipynb (sección 10.3) con el modelo congelado.")
    meta = json.loads(stem.with_suffix(".json").read_text(encoding="utf-8"))
    dets = [None if d is None else {"boxes": np.asarray(d["boxes"], np.float32).reshape(-1, 4),
                                    "scores": np.asarray(d["scores"], np.float32), "labels": np.asarray(d["labels"], np.int64)}
            for d in meta["dets"]]
    return dets, np.load(stem.with_suffix(".npz"), mmap_mode="r")["sem"]


def boxes_from_gt(sem256):
    boxes, labels = [], []
    for k in (1, 2, 3):
        ys, xs = np.nonzero(sem256 == k)
        if len(ys):
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]); labels.append(k)
    return {"boxes": np.asarray(boxes, np.float32).reshape(-1, 4), "labels": np.asarray(labels, np.int64),
            "scores": np.ones(len(labels), np.float32)}


def sam_semantic(image256, det, predictor):
    """SAM con las cajas dadas -> máscara semántica 256 x 256 (0 fondo, 1..3 región de la caja)."""
    semantic = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    if det is None or len(det["boxes"]) == 0:
        return semantic
    instances, classes, _ = run_sam_on_predicted_boxes(image256, det, predictor)
    for instance_id, class_id in classes.items():
        semantic[instances == instance_id] = class_id
    return semantic


def run_sam_case(case_id, predictor, box_source="pred", keep_examples=6):
    volume, label = load_case(case_id)
    dets, model_sem = (load_predicted_boxes(case_id) if box_source == "pred" else (None, None))
    positives = np.flatnonzero((label > 0).any(axis=(1, 2)))[::SLICE_STRIDE]      # cortes con hueso en la referencia
    if MAX_SLICES_PER_CASE:
        positives = positives[np.linspace(0, len(positives) - 1, min(MAX_SLICES_PER_CASE, len(positives))).astype(int)]
    acc_sam, acc_model = RegionAccumulator(), RegionAccumulator()
    sam_volume = np.zeros((volume.shape[0], IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    rows, examples, seconds = [], [], []
    for z in positives:
        image256, gt256, _ = letterbox_pair(enhance_bone_contrast(apply_hu_window(volume[z])), label[z], IMAGE_SIZE)
        gt_sem = instance_to_semantic(gt256)
        det = dets[z] if box_source == "pred" else boxes_from_gt(gt_sem)
        if det is not None and box_source == "pred":
            keep = det["scores"] >= BOX_SCORE_MIN
            det = {k: v[keep] for k, v in det.items()}
        t0 = time.time()
        sam_sem = sam_semantic(image256, det, predictor)
        seconds.append(time.time() - t0)
        sam_volume[z] = sam_sem
        acc_sam.add(sam_sem, gt_sem)
        if box_source == "pred":
            mod_sem = np.asarray(model_sem[z]) if (det is not None and len(det["boxes"])) else np.zeros_like(gt_sem)
            acc_model.add(mod_sem, gt_sem)
        else:
            mod_sem = None
        present = [k for k in (1, 2, 3) if (gt_sem == k).any()]
        dice = lambda p: float(np.mean([2 * ((p == k) & (gt_sem == k)).sum() / max(((p == k).sum() + (gt_sem == k).sum()), 1) for k in present]))
        rows.append({"caso": case_id, "corte": int(z), "n_cajas": 0 if det is None else len(det["boxes"]),
                     "dice_sam": dice(sam_sem), "dice_modelo": dice(mod_sem) if mod_sem is not None else np.nan})
        examples.append((rows[-1]["dice_sam"], z, image256.astype(np.float16), gt_sem, mod_sem, sam_sem, det))
    examples.sort(key=lambda e: e[0])
    kept = examples[:keep_examples] + examples[-3:]
    return {"sam": acc_sam, "modelo": acc_model, "slices": pd.DataFrame(rows), "sam_volume": sam_volume,
            "examples": kept, "seconds_per_slice": float(np.mean(seconds)) if seconds else np.nan}


def plot_examples(examples, title, path=None):
    fig, axes = plt.subplots(len(examples), 4, figsize=(11, 2.8 * len(examples)), squeeze=False)
    cmap = np.array([[0, 0, 0, 0], [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[1], 0.7)],
                     [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[2], 0.7)], [*plt.matplotlib.colors.to_rgba(TARGET_COLORS[3], 0.7)]])
    for r, (dice_sam, z, img, gt, mod, sam, det) in enumerate(examples):
        for c, (mask, name) in enumerate(((None, "cajas predichas"), (gt, "referencia"), (mod, "modelo propio"), (sam, f"SAM (Dice {dice_sam:.2f})"))):
            ax = axes[r, c]; ax.imshow(img.astype(np.float32), cmap="gray"); ax.axis("off")
            if mask is not None:
                ax.imshow(cmap[mask])
            if c == 0 and det is not None:
                for b in det["boxes"]:
                    ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1.2))
            ax.set_title(f"{name} · corte {z}" if c == 0 else name, fontsize=8)
    fig.suptitle(title); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=140)
    return fig


## 3. Ejecución: SAM vs modelo propio

La tabla compara las dos segmentaciones sobre los mismos cortes y con las mismas cajas. En el modelo propio, un corte sin cajas queda vacío (igual que en SAM). Se guardan las máscaras de SAM por caso (`evidencias/sam/sam_<split>_<caso>.npz`, matriz `[corte, 256, 256]` con 0 fondo y 1/2/3 región, más qué cortes se procesaron) para que Hoyos calcule las métricas finales, y el Dice por corte (`dice_por_corte_<split>.csv`).


In [ ]:
predictor = create_sam_predictor(SAM_CHECKPOINT, SAM_MODEL_TYPE, DEVICE)       # falla con un mensaje claro si falta la librería o el checkpoint

results, all_slices, examples_all = {}, [], []
accumulators = {"SAM (cajas predichas)": RegionAccumulator(), "Modelo propio (mismas cajas)": RegionAccumulator()}
for case_id in CASES[SPLIT]:
    res = run_sam_case(case_id, predictor, "pred")
    results[case_id] = res
    for key, name in (("sam", "SAM (cajas predichas)"), ("modelo", "Modelo propio (mismas cajas)")):
        for k in (1, 2, 3):
            for j in range(3):
                accumulators[name].c[k][j] += res[key].c[k][j]
    all_slices.append(res["slices"]); examples_all += res["examples"]
    np.savez_compressed(SAM_DIR / f"sam_{SPLIT}_{case_id}.npz", sam_semantic=res["sam_volume"],
                        processed=(res["sam_volume"].reshape(len(res["sam_volume"]), -1).any(axis=1)))
    print(f"caso {case_id}: {len(res['slices'])} cortes | {res['seconds_per_slice']:.2f} s por corte con SAM ({DEVICE.type}) | "
          f"Dice SAM {res['sam'].summary()['dice_medio']:.3f} vs modelo {res['modelo'].summary()['dice_medio']:.3f}")

summary = pd.DataFrame({name: acc.summary() for name, acc in accumulators.items()}).T
summary.to_csv(SAM_DIR / f"comparacion_sam_vs_modelo_{SPLIT}.csv")
display(summary.round(3))
slices_df = pd.concat(all_slices, ignore_index=True)
slices_df.to_csv(SAM_DIR / f"dice_por_corte_{SPLIT}.csv", index=False)
print(f"Latencia SAM por corte: {np.mean([r['seconds_per_slice'] for r in results.values()]):.2f} s ({DEVICE.type}, {SAM_MODEL_TYPE})")


## 4. Galería de los peores cortes de SAM

Sirve para el informe: cada caso fallido debe ir acompañado de una hipótesis de causa.


In [ ]:
worst = sorted(examples_all, key=lambda e: e[0])[:5]
plot_examples(worst, f"Los 5 cortes con peor Dice de SAM ({SPLIT})", SAM_DIR / f"peores_sam_{SPLIT}.png"); plt.show()
print("Hipótesis a contrastar al mirar la galería: (1) SAM segmenta TODA la región ósea dentro de la caja y no separa fragmentos; "
      "(2) el contraste Otsu+CLAHE de la imagen no es el de las fotos naturales con que se entrenó SAM; "
      "(3) cajas que cubren dos huesos (sacro y coxal se tocan en la articulación) producen máscaras que se invaden.")


## 5. Cota superior con cajas reales (opcional)

No es el protocolo de la consigna. Indica cuánto del error de SAM viene de las cajas mal predichas y cuánto de SAM mismo.


In [ ]:
if ALSO_GT_BOXES:
    gt_box_acc = RegionAccumulator()
    for case_id in CASES[SPLIT]:
        run = run_sam_case(case_id, predictor, "gt")
        for k in (1, 2, 3):
            for j in range(3):
                gt_box_acc.c[k][j] += run["sam"].c[k][j]
    display(pd.DataFrame({"SAM (cajas REALES, cota superior)": gt_box_acc.summary()}).T.round(3))
else:
    print("ALSO_GT_BOXES = False: no se calcula la cota con cajas reales.")


## 6. Qué reportar

Tabla de la sección 3 (Dice e IoU por región, SAM vs modelo propio), latencia por corte de SAM (CPU o GPU), la galería y la discusión: SAM recibe una caja por región y no separa fragmentos, así que no puede igualar el nivel de instancia del modelo propio; la comparación relevante es la de región.
